In [1]:
import psycopg2
import psycopg2.extras
import yaml
import os
import logging
from typing import Dict, Any, List, Optional
from contextlib import contextmanager
from string import Template
from pathlib import Path

In [4]:
class DatabaseClient:
    """데이터베이스 연결 및 데이터 조회를 담당하는 클라이언트 클래스"""
    
    def __init__(self, config_path: str = "config/settings.yaml"):
        """
        데이터베이스 클라이언트 초기화
        
        Args:
            config_path: 설정 파일 경로
        """
        self.config = self._load_config(config_path)
        self.connection = None
        self.logger = self._setup_logger()
    
    def _load_config(self, config_path: str) -> Dict[str, Any]:
        """설정 파일 로드 및 환경 변수 치환"""
        try:
            with open(config_path, 'r', encoding='utf-8') as file:
                content = file.read()
                # 환경 변수 치환
                template = Template(content)
                substituted_content = template.substitute(os.environ)
                return yaml.safe_load(substituted_content)
        except FileNotFoundError:
            raise FileNotFoundError(f"설정 파일을 찾을 수 없습니다: {config_path}")
        except yaml.YAMLError as e:
            raise ValueError(f"설정 파일 파싱 오류: {e}")
        except KeyError as e:
            raise ValueError(f"환경 변수가 설정되지 않았습니다: {e}")
    
    def _setup_logger(self) -> logging.Logger:
        """로거 설정"""
        logger = logging.getLogger(__name__)
        if not logger.handlers:
            # 로그 폴더 생성
            log_dir = Path("logs")
            log_dir.mkdir(exist_ok=True)
            
            # 파일 핸들러
            file_handler = logging.FileHandler(log_dir / "db_client.log", encoding='utf-8')
            file_formatter = logging.Formatter(
                '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
            )
            file_handler.setFormatter(file_formatter)
            file_handler.setLevel(logging.INFO)
            logger.addHandler(file_handler)
            
            # 콘솔 핸들러
            console_handler = logging.StreamHandler()
            console_formatter = logging.Formatter(
                self.config.get('logging', {}).get('format', 
                    '%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            )
            console_handler.setFormatter(console_formatter)
            console_handler.setLevel(logging.INFO)
            logger.addHandler(console_handler)
            
            logger.setLevel(
                getattr(logging, 
                    self.config.get('logging', {}).get('level', 'INFO'))
            )
        return logger
    
    def connect(self) -> None:
        """데이터베이스 연결"""
        try:
            db_config = self.config['database']
            self.connection = psycopg2.connect(
                host=db_config['host'],
                port=db_config['port'],
                user=db_config['username'],
                password=REDACTED'password'],
                database=db_config['database_name'],
                connect_timeout=10
            )
            self.logger.info("데이터베이스 연결 성공")
        except psycopg2.Error as e:
            self.logger.error(f"데이터베이스 연결 실패: {e}")
            raise
    
    def disconnect(self) -> None:
        """데이터베이스 연결 해제"""
        if self.connection:
            try:
                self.connection.close()
                self.logger.info("데이터베이스 연결 해제")
            except psycopg2.Error as e:
                self.logger.error(f"데이터베이스 연결 해제 중 오류: {e}")
            finally:
                self.connection = None

In [9]:
config_path = "config/settings.yaml"
db_client = DatabaseClient(config_path)
db_client.connect()

2026-03-19 11:11:30,482 - __main__ - INFO - 데이터베이스 연결 성공


In [14]:
cursor = db_client.connection.cursor()
sql01 = """
SELECT 
    device_id
FROM site_devices
WHERE site_id = 'ags_003'
"""
sql02 = """
SELECT
    DISTINCT facility_id
FROM facility_devices
WHERE device_id IN %s
"""
sql03 = """
SELECT
    DISTINCT facility_id
FROM facility_controls
WHERE device_id IN %s
"""
cursor.execute(sql01)
results = cursor.fetchall()
print(results)

# Extract device_ids from results
device_ids = [result[0] for result in results]

# Use tuple for IN clause
device_ids_tuple = tuple(device_ids) if device_ids else tuple([''])  # Handle empty case

cursor.execute(sql02, (device_ids_tuple,))
results02 = cursor.fetchall()

cursor.execute(sql03, (device_ids_tuple,))
results03 = cursor.fetchall()

print("Device IDs:", device_ids)
print("Facility Devices:", results02)
print("Facility Controls:", results03)

[('0D2509224001',), ('0D2509224002',)]
Device IDs: ['0D2509224001', '0D2509224002']
Facility Devices: [('UUID_000011',), ('UUID_000012',), ('UUID_000013',), ('UUID_000014',), ('UUID_000015',), ('UUID_000016',), ('UUID_000017',), ('UUID_000018',)]
Facility Controls: [('UUID_000013',), ('UUID_000014',), ('UUID_000011',), ('UUID_000012',), ('UUID_000015',), ('UUID_000016',), ('UUID_000017',), ('UUID_000018',)]


In [15]:
total_facilities = set(results02 + results03)
print(f"Total facilities: {len(total_facilities)}")
print(sorted(total_facilities))

Total facilities: 8
[('UUID_000013',), ('UUID_000017',), ('UUID_000015',), ('UUID_000016',), ('UUID_000014',), ('UUID_000011',), ('UUID_000012',), ('UUID_000018',)]


In [12]:
sql02 = """
SELECT
    facility_id
FROM facility_devices
WHERE device_id IN ('0D2509224001', '0D2509224002')
"""
cursor.execute(sql02)
print(cursor.fetchall())

[('UUID_000015',), ('UUID_000018',), ('UUID_000014',), ('UUID_000012',), ('UUID_000017',), ('UUID_000016',), ('UUID_000011',), ('UUID_000013',), ('UUID_000015',), ('UUID_000018',), ('UUID_000014',), ('UUID_000012',), ('UUID_000017',), ('UUID_000016',), ('UUID_000011',), ('UUID_000013',)]
